# Reproducibility: Environments, Docker, Config and Seeds

Notebook for **Chapter 2** of the *mlops* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/mlops/ch02.html

Tip: change a number, predict the output, then run the cell again.

## 2.1 Seeds and Randomness

In [ ]:
import hashlib, json, sys, sklearn, numpy
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
def run(seed=None):
    X, y = make_classification(n_samples=600, n_features=10, n_informative=4, random_state=7)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=7)
    m = RandomForestClassifier(n_estimators=20, max_depth=4, random_state=seed).fit(Xtr, ytr)
    return round(m.score(Xte, yte), 4)
print("seed=1 :", run(1), run(1))
print("seed=2 :", run(2))
print("no seed:", run(None), run(None), run(None))

## 2.2 Environments, Dependencies and Lock Files

In [ ]:
import sys, sklearn, numpy
print("env:", sys.version.split()[0], "numpy", numpy.__version__, "sklearn", sklearn.__version__)

env: 3.14.3 numpy 2.5.3 sklearn 1.9.1


## 2.3 Configuration Management

In [ ]:
from dataclasses import dataclass
import yaml
text = """
model:
  name: logreg
  C: 0.5
train:
  seed: 42
  test_size: 0.2
"""
@dataclass(frozen=True)
class TrainCfg:
    seed: int
    test_size: float
@dataclass(frozen=True)
class ModelCfg:
    name: str
    C: float
raw = yaml.safe_load(text)
m = ModelCfg(**raw["model"]); t = TrainCfg(**raw["train"])
print(m); print(t)
try:
    TrainCfg(**{"seed": 1, "test_sizee": 0.2})
except TypeError as e:
    print("typo caught:", e)

In [ ]:
import hashlib, json
cfg = {"model": "rf", "n_estimators": 20, "max_depth": 4, "seed": 1, "data": "v3"}
print("config hash:", hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:12])
cfg["max_depth"] = 5
print("changed hash:", hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:12])

config hash: 0c0f76eeb44f
changed hash: 6bded236377e


## 2.4 Docker for ML

## 2.5 The Five Ingredients and the Run Manifest

The first four sections gave you four tools. Before we go deeper into each, we need one map that tells you where to look when something goes wrong. Every training run is a function with five inputs. If you can name and record all five, you can reproduce the run. If two runs disagree, at least one of the five differs, and the rest of this chapter is about finding which one quickly.

In [ ]:
import hashlib, json, platform
import numpy as np, sklearn
def sha(b): return hashlib.sha256(b).hexdigest()[:12]
data = b"id,x,y\n1,0.5,1\n2,0.7,0\n"
cfg = {"lr": 0.1, "seed": 1}
manifest = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "sklearn": sklearn.__version__,
    "data_sha": sha(data),
    "config_sha": sha(json.dumps(cfg, sort_keys=True).encode()),
    "seed": cfg["seed"],
}
print(json.dumps(manifest, indent=2))

{
  "python": "3.14.3",
  "numpy": "2.5.3",
  "sklearn": "1.9.1",
  "data_sha": "dc64c3b65a7d",
  "config_sha": "2ccbe222bdf9",
  "seed": 1
}


## 2.6 Seeds in Depth: Every Source of Randomness

Section 2.1 showed that a seed locks a run. In practice a project has many random number generators (RNGs), and each has its own state. Seeding one does not seed the others. This section catalogues them with real examples so that you can find the one you forgot.

In [ ]:
import random
import numpy as np
random.seed(5); a = [round(random.random(), 4) for _ in range(2)]
np.random.seed(5); b = np.random.rand(2).round(4)
c = np.random.default_rng(5).random(2).round(4)
print("python random    :", a)
print("numpy legacy     :", b)
print("numpy Generator  :", c)
random.seed(5)
np.random.seed(0)
print("after random.seed(5), numpy legacy draw:", np.random.rand(1).round(4))

python random    : [0.6229, 0.7418]
numpy legacy     : [0.222  0.8707]
numpy Generator  : [0.805  0.8079]
after random.seed(5), numpy legacy draw: [0.5488]


In [ ]:
import subprocess, sys
code = "print(hash('mlops') % 1000, list({'apple','pear','fig'}))"
for s in ("0", "0", "1"):
    out = subprocess.run([sys.executable, "-c", code], capture_output=True,
                         text=True, env={"PYTHONHASHSEED": s}).stdout.strip()
    print("PYTHONHASHSEED=" + s, out)

PYTHONHASHSEED=0 491 ['pear', 'apple', 'fig']
PYTHONHASHSEED=0 491 ['pear', 'apple', 'fig']
PYTHONHASHSEED=1 140 ['pear', 'fig', 'apple']


In [ ]:
import numpy as np
for w in range(3):
    print("naive worker", w, np.random.default_rng(42).integers(0, 100, 3))
ss = np.random.SeedSequence(2024)
for i, k in enumerate(ss.spawn(3)):
    print("child", i, np.random.default_rng(k).random(2).round(4))
print("child 1 again", np.random.default_rng(np.random.SeedSequence(2024).spawn(3)[1]).random(2).round(4))

naive worker 0 [ 8 77 65]
naive worker 1 [ 8 77 65]
naive worker 2 [ 8 77 65]
child 0 [0.6506 0.5296]
child 1 [0.1238 0.4583]
child 2 [0.9212 0.1295]
child 1 again [0.1238 0.4583]


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
X, y = make_classification(n_samples=600, n_features=10, n_informative=4, random_state=7)
def score(depth, seed):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed)
    return RandomForestClassifier(n_estimators=20, max_depth=depth, random_state=seed).fit(Xtr, ytr).score(Xte, yte)
a = np.array([score(4, s) for s in range(10)]); b = np.array([score(6, s) for s in range(10)])
d = b - a
print("A  mean %.4f std %.4f" % (a.mean(), a.std(ddof=1)))
print("B  mean %.4f std %.4f" % (b.mean(), b.std(ddof=1)))
print("paired diff mean %.4f std %.4f" % (d.mean(), d.std(ddof=1)))
print("standard error of diff %.4f" % (d.std(ddof=1) / np.sqrt(10)))
print("B wins in", int((d > 0).sum()), "of 10 seeds; ties", int((d == 0).sum()))

A  mean 0.8850 std 0.0257
B  mean 0.8933 std 0.0194
paired diff mean 0.0083 std 0.0144
standard error of diff 0.0046
B wins in 6 of 10 seeds; ties 1


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
X, y = make_classification(n_samples=600, n_features=10, n_informative=4, random_state=7)
def sc(split_seed, model_seed):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=split_seed)
    return RandomForestClassifier(n_estimators=20, max_depth=4, random_state=model_seed).fit(Xtr, ytr).score(Xte, yte)
v_model = [sc(0, s) for s in range(10)]
v_split = [sc(s, 0) for s in range(10)]
print("model seed varies, split fixed: std %.4f" % np.std(v_model, ddof=1))
print("split seed varies, model fixed: std %.4f" % np.std(v_split, ddof=1))

model seed varies, split fixed: std 0.0136
split seed varies, model fixed: std 0.0245


## 2.7 Floating-Point Order and Hidden Non-Determinism

Suppose you did everything in section 2.6 right: all seeds fixed, same data, same environment. Two runs can still differ in the last digits, especially with parallel code or GPUs. The cause is not randomness at all. It is how computers add decimal numbers.

In [ ]:
a, b, c = 0.1, 0.2, 0.3
print(a + b)
print(a + b == c)
print((a + b) + c == a + (b + c))
print((1e16 + 1.0) - 1e16, 1e16 + (1.0 - 1e16))

0.30000000000000004
False
False
0.0 0.0


In [ ]:
import math
import numpy as np
from functools import reduce
from operator import add
rng = np.random.default_rng(0)
values = (rng.normal(size=100_000) * 10.0 ** rng.integers(-3, 8, size=100_000)).tolist()
ref = math.fsum(values)
errs = []
for t in range(300):
    o = values[:]
    np.random.default_rng(t).shuffle(o)
    errs.append(reduce(add, o) - ref)
errs = np.array(errs)
print("distinct sums among 300 orders:", len(set(errs.tolist())))
print("std of error: %.2e" % errs.std())
print("worst error : %.2e" % np.abs(errs).max())
print("pairwise np.sum error     : %.2e" % (np.sum(np.array(values)) - ref))
print("left-to-right, original   : %.2e" % (reduce(add, values) - ref))

distinct sums among 300 orders: 256
std of error: 4.92e-06
worst error : 2.03e-05
pairwise np.sum error     : 8.94e-08
left-to-right, original   : 4.38e-06


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
def train(seed):
    X, y = make_classification(n_samples=200, n_features=5, random_state=seed)
    m = LogisticRegression(max_iter=500).fit(X, y)
    return m.coef_.ravel()
w1, w2 = train(3), train(3)
print("bit identical :", np.array_equal(w1, w2))
print("allclose      :", np.allclose(w1, w2, rtol=1e-9, atol=1e-12))
print("max abs diff  :", float(np.abs(w1 - w2).max()))
print("different seed close? :", np.allclose(w1, train(4), rtol=1e-3))

bit identical : True
allclose      : True
max abs diff  : 0.0
different seed close? : False


## 2.8 Environments in Practice: venv, pip, Lock Files and uv

Section 2.2 explained why lock files matter. Here is the working routine, with the commands you will type, what each one does, and how to check a result in code. Only `venv`, `pip` and the Python standard library are available on this machine; `pip-compile` and `uv` are shown as correct commands labelled *not run here*.

In [ ]:
from importlib.metadata import version
for name in ["numpy", "pandas", "scikit-learn", "PyYAML"]:
    print(f"{name}=={version(name)}")

numpy==2.5.3
pandas==3.0.6
scikit-learn==1.9.1
PyYAML==6.0.3


In [ ]:
def parse(text):
    out = {}
    for line in text.strip().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "==" in line:
            name, ver = line.split("==")
            out[name.lower().replace("_", "-")] = ver
    return out
def diff(good, bad):
    g, b = parse(good), parse(bad)
    for k in sorted(set(g) | set(b)):
        if g.get(k) != b.get(k):
            print(f"{k:14s} good={g.get(k, '-'):8s} bad={b.get(k, '-')}")
good = """numpy==2.5.3
scikit-learn==1.9.1
pandas==3.0.6
joblib==1.5.2
Pillow==11.0.0"""
bad = """numpy==2.5.3
scikit_learn==1.9.1
pandas==2.2.3
joblib==1.5.2
tqdm==4.67.1"""
diff(good, bad)

pandas         good=3.0.6    bad=2.2.3
pillow         good=11.0.0   bad=-
tqdm           good=-        bad=4.67.1


In [ ]:
import re
reqs = """numpy==2.5.3
pandas>=2.0
scikit-learn
PyYAML==6.0.3
requests~=2.32
"""
bad = []
for line in reqs.strip().splitlines():
    if not re.fullmatch(r"[A-Za-z0-9_.\-]+==[A-Za-z0-9_.+!\-]+", line.strip()):
        bad.append(line)
print("unpinned:", bad)
print("OK" if not bad else "FAIL: pin these exactly")

unpinned: ['pandas>=2.0', 'scikit-learn', 'requests~=2.32']
FAIL: pin these exactly


In [ ]:
import hashlib
wheel_bytes = b"pretend these are the bytes of numpy-2.5.3.whl"
locked = hashlib.sha256(wheel_bytes).hexdigest()
downloaded_ok = wheel_bytes
downloaded_bad = wheel_bytes + b"!"
print("lock hash :", locked[:16])
print("good file :", hashlib.sha256(downloaded_ok).hexdigest() == locked)
print("tampered  :", hashlib.sha256(downloaded_bad).hexdigest() == locked)

lock hash : 91130851ff9753d9
good file : True
tampered  : False


## 2.9 Configuration in Depth: Layers, Overrides and Hydra-Style Composition

Section 2.3 showed typed configs. Real projects build the final configuration from several layers: defaults, an environment file (dev or prod), an experiment file, and command-line overrides. Hydra automates this; the underlying idea is a deep merge, and you can implement it in a dozen lines. That makes the idea concrete and testable.

In [ ]:
import copy, json
def merge(base, over):
    out = copy.deepcopy(base)
    for k, v in over.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = merge(out[k], v)
        else:
            out[k] = v
    return out
def parse_override(cfg, text):
    path, raw = text.split("=", 1)
    keys = path.split(".")
    node = cfg
    for k in keys[:-1]:
        node = node.setdefault(k, {})
    try:
        node[keys[-1]] = json.loads(raw)
    except ValueError:
        node[keys[-1]] = raw
    return cfg
defaults = {"model": {"name": "rf", "depth": 4}, "train": {"lr": 0.1, "seed": 1}}
exp_file = {"model": {"depth": 6}}
cfg = merge(defaults, exp_file)
for o in ["train.seed=7", "train.lr=0.05", "model.name=gbm"]:
    parse_override(cfg, o)
print(json.dumps(cfg, sort_keys=True))
print("defaults untouched:", defaults["model"]["depth"], defaults["train"]["seed"])

{"model": {"depth": 6, "name": "gbm"}, "train": {"lr": 0.05, "seed": 7}}
defaults untouched: 4 1


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
from dataclasses import dataclass
import hashlib, json
@dataclass(frozen=True)
class Train:
    lr: float
    seed: int
    epochs: int = 10
    def __post_init__(self):
        if not (0 < self.lr < 1):
            raise ValueError(f"lr must be in (0,1), got {self.lr}")
        if self.seed < 0:
            raise ValueError("seed must be >= 0")
print(Train(lr=0.05, seed=7))
for bad in [dict(lr=5, seed=1), dict(lr="0.1", seed=1)]:
    try:
        print(Train(**bad))
    except (ValueError, TypeError) as e:
        print(type(e).__name__, e)
cfg_a = {"train": {"lr": 0.05, "seed": 7}, "model": {"depth": 6}}
cfg_b = {"model": {"depth": 6}, "train": {"seed": 7, "lr": 0.05}}
h = lambda c: hashlib.sha256(json.dumps(c, sort_keys=True).encode()).hexdigest()[:12]
print(h(cfg_a), h(cfg_b), h(cfg_a) == h(cfg_b))
print(h({"train": {"lr": 0.05, "seed": 7}, "model": {"depth": "6"}}), "(depth given as a string)")
print(h({"train": {"lr": 0.05, "seed": 7.0}, "model": {"depth": 6}}), "(seed given as 7.0)")

Train(lr=0.05, seed=7, epochs=10)
ValueError lr must be in (0,1), got 5
TypeError '<' not supported between instances of 'int' and 'str'
0f2a28d4606c 0f2a28d4606c True
8b1b69daa066 (depth given as a string)
f1de088fc9ad (seed given as 7.0)


## 2.10 Docker in Depth: Layers, Digests and Size

Section 2.4 gave a good Dockerfile and the reasons for each line. Now we make caching and size concrete. Docker is not installed here, so all Docker commands are labelled *not run here*; the caching model, however, is simple enough to simulate in Python and see the effect with real numbers.

In [ ]:
import hashlib
def key(parent, instr, content=""):
    return hashlib.sha256((parent + "|" + instr + "|" + content).encode()).hexdigest()[:8]
def build(files, order, cache):
    parent, total = "", 0
    cost = {"FROM": 0, "COPY lock": 0, "RUN pip install": 240, "COPY code": 1, "COPY all": 2}
    for step in order:
        content = {"COPY lock": files["lock"], "COPY code": files["code"],
                   "COPY all": files["lock"] + files["code"]}.get(step, "")
        k = key(parent, step, content)
        hit = k in cache
        cache.add(k)
        t = 0 if hit else cost[step]
        total += t
        print(f"  {step:16s} {'CACHED' if hit else 'rebuilt':8s} {t:4d} s")
        parent = k
    print(f"  total {total} s")
good = ["FROM", "COPY lock", "RUN pip install", "COPY code"]
bad = ["FROM", "COPY all", "RUN pip install"]
for name, order in [("good order", good), ("bad order", bad)]:
    cache = set()
    files = {"lock": "numpy==2.5.3", "code": "v1"}
    print(name, "- first build"); build(files, order, cache)
    files["code"] = "v2"
    print(name, "- after editing one line of code"); build(files, order, cache)
    files["lock"] = "numpy==2.5.4"
    print(name, "- after bumping a dependency"); build(files, order, cache)

good order - first build
  FROM             rebuilt     0 s
  COPY lock        rebuilt     0 s
  RUN pip install  rebuilt   240 s
  COPY code        rebuilt     1 s
  total 241 s
good order - after editing one line of code
  FROM             CACHED      0 s
  COPY lock        CACHED      0 s
  RUN pip install  CACHED      0 s
  COPY code        rebuilt     1 s
  total 1 s
good order - after bumping a dependency
  FROM             CACHED      0 s
  COPY lock        rebuilt     0 s
  RUN pip install  rebuilt   240 s
  COPY code        rebuilt     1 s
  total 241 s
bad order - first build
  FROM             rebuilt     0 s
  COPY all         rebuilt     2 s
  RUN pip install  rebuilt   240 s
  total 242 s
bad order - after editing one line of code
  FROM             CACHED      0 s
  COPY all         rebuilt     2 s
  RUN pip install  rebuilt   240 s
  total 242 s
bad order - after bumping a dependency
  FROM             CACHED      0 s
  COPY all         rebuilt     2 s
  RUN pip install

## 2.11 Hashing Data and Code: Fingerprints You Can Trust

Hashes appeared in sections 2.3, 2.5 and 2.8. This section gives them a proper treatment because they are the cheapest tool for the question "did anything change?". A hash function reads any amount of data and produces a fixed-size fingerprint. SHA-256 gives 256 bits (64 hex characters). The same input always yields the same output; a different input yields a completely different output with overwhelming probability.

In [ ]:
import hashlib, io
def file_sha(f, chunk=4):
    h = hashlib.sha256()
    while True:
        b = f.read(chunk)
        if not b:
            break
        h.update(b)
    return h.hexdigest()[:12]
raw = b"id,x\n1,0.5\n2,0.7\n"
print("chunked :", file_sha(io.BytesIO(raw), chunk=4))
print("one shot:", hashlib.sha256(raw).hexdigest()[:12])
print("trailing newline removed:", hashlib.sha256(raw[:-1]).hexdigest()[:12])
print("CRLF line endings       :", hashlib.sha256(raw.replace(b"\n", b"\r\n")).hexdigest()[:12])

chunked : 2dc12095afc4
one shot: 2dc12095afc4
trailing newline removed: 22bb0dc1b1fb
CRLF line endings       : cd1658715ece


In [ ]:
import hashlib
import numpy as np
import pandas as pd
a = np.array([1, 2, 3])
print(hashlib.sha256(a.tobytes()).hexdigest()[:12], a.dtype)
print(hashlib.sha256(a.astype(np.int32).tobytes()).hexdigest()[:12], "int32")
print(hashlib.sha256(a.astype(float).tobytes()).hexdigest()[:12], "float64")
df = pd.DataFrame({"id": [1, 2, 3], "x": [0.5, 0.7, 0.9]})
def df_sha(d):
    return hashlib.sha256(pd.util.hash_pandas_object(d, index=True).values.tobytes()).hexdigest()[:12]
print("original      :", df_sha(df))
print("rows reversed :", df_sha(df.iloc[::-1]))
print("sorted back   :", df_sha(df.iloc[::-1].sort_values("id")))
print("column order  :", df_sha(df[["x", "id"]]))
df2 = df.copy(); df2.loc[1, "x"] = 0.7000001
print("tiny edit     :", df_sha(df2))

e2e2033ae7e1 int64
4636993d3e1d int32
a68de4b5e96a float64
original      : 684e07add8d2
rows reversed : dcbdbcc2fbcc
sorted back   : 684e07add8d2
column order  : 2b6465ac070e
tiny edit     : feedb92ceec4


In [ ]:
import hashlib, json
base = {"lr": 0.1, "depth": 4, "seed": 1}
def H(c): return hashlib.sha256(json.dumps(c, sort_keys=True).encode()).hexdigest()
def bits(a, b): return bin(int(a, 16) ^ int(b, 16)).count("1")
h0 = H(base)
flips = []
for i in range(1000):
    c = dict(base); c["lr"] = 0.1 + (i + 1) * 1e-9
    flips.append(bits(h0, H(c)))
print("mean bits flipped of 256: %.1f, min %d, max %d" % (sum(flips) / len(flips), min(flips), max(flips)))

mean bits flipped of 256: 127.7, min 101, max 154


In [ ]:
import hashlib
files = {"train.py": b"print('train')\n", "data/a.csv": b"id,x\n1,0.5\n2,0.7\n", "cfg.yaml": b"lr: 0.1\n"}
def tree_sha(fs):
    lines = [f"{hashlib.sha256(b).hexdigest()[:8]}  {n}" for n, b in sorted(fs.items())]
    return hashlib.sha256("\n".join(lines).encode()).hexdigest()[:12], lines
top, lines = tree_sha(files)
print("top:", top)
print("\n".join(lines))
files["cfg.yaml"] = b"lr: 0.2\n"
top2, lines2 = tree_sha(files)
print("top after editing cfg.yaml:", top2)
print("changed:", [a.split()[1] for a, b in zip(lines, lines2) if a != b])

top: f85ddbf206a6
d245e06b  cfg.yaml
2dc12095  data/a.csv
c7d9b938  train.py
top after editing cfg.yaml: 0d58c9775f5f
changed: ['cfg.yaml']


## 2.12 How to Solve Any Reproducibility Problem

You now have the pieces: five ingredients, seeds, floating-point order, locked environments, configs, containers and hashes. This section turns them into a procedure. Reproducibility problems look different on the surface ("the metric moved", "it crashes only on the server", "the Docker build is slow", "the model I shipped is not the model I tested"), but they all reduce to the same question: *which of the five inputs differs, and can I prove it?*

In [ ]:
import hashlib
import numpy as np
def level(a, b, rtol=1e-6, atol=1e-9):
    if a.shape != b.shape:
        return "different shape"
    if a.dtype == b.dtype and hashlib.sha256(a.tobytes()).digest() == hashlib.sha256(b.tobytes()).digest():
        return "bit-identical"
    if np.allclose(a, b, rtol=rtol, atol=atol):
        return "numerically close"
    return "different"
x = np.array([0.1, 0.2, 0.3, 0.4, 0.5])
print("copy     :", level(x, x.copy()))
print("+1e-12   :", level(x, x + 1e-12))
print("float32  :", level(x, x.astype(np.float32).astype(np.float64)))
print("+1e-3    :", level(x, x + 1e-3))
print("shape    :", level(x, x[:4]))

copy     : bit-identical
+1e-12   : numerically close
float32  : numerically close
+1e-3    : different
shape    : different shape


In [ ]:
import hashlib, numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
def H(a): return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()[:8]
def pipeline(split_seed, model_seed):
    X, y = make_classification(n_samples=300, n_features=8, random_state=0)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=split_seed)
    m = RandomForestClassifier(n_estimators=10, random_state=model_seed).fit(Xtr, ytr)
    return {"split": H(Xte), "fit": H(m.feature_importances_), "predict": H(m.predict_proba(Xte))}
def compare(fn, *args):
    r1, r2 = fn(*args), fn(*args)
    return {k: ("same" if r1[k] == r2[k] else "DIFFERENT") for k in r1}
for name, args in [("all seeded", (3, 3)), ("model unseeded", (3, None)), ("split unseeded", (None, 3))]:
    print(f"{name:15s}", compare(pipeline, *args))

all seeded      {'split': 'same', 'fit': 'same', 'predict': 'same'}
model unseeded  {'split': 'same', 'fit': 'DIFFERENT', 'predict': 'DIFFERENT'}
split unseeded  {'split': 'DIFFERENT', 'fit': 'DIFFERENT', 'predict': 'DIFFERENT'}


In [ ]:
def parse(text):
    return dict(l.strip().lower().replace("_", "-").split("==") for l in text.strip().splitlines() if "==" in l)
last_month = """numpy==2.5.3
pandas==3.0.5
scikit-learn==1.9.1
joblib==1.5.2"""
today = """numpy==2.5.3
pandas==3.0.6
scikit-learn==1.9.2
joblib==1.5.2
threadpoolctl==3.6.0"""
a, b = parse(last_month), parse(today)
changed = {k: (a.get(k), b.get(k)) for k in sorted(set(a) | set(b)) if a.get(k) != b.get(k)}
print(len(changed), "differences")
for k, (old, new) in changed.items():
    print(f"  {k:14s} {old} -> {new}")

3 differences
  pandas         3.0.5 -> 3.0.6
  scikit-learn   1.9.1 -> 1.9.2
  threadpoolctl  None -> 3.6.0


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import hashlib
def load_verified(raw, expected):
    got = hashlib.sha256(raw).hexdigest()[:12]
    if got != expected:
        raise RuntimeError(f"data changed: expected {expected}, got {got}")
    return raw.decode().splitlines()
good = b"id,x\n1,0.5\n2,0.7\n"
expected = hashlib.sha256(good).hexdigest()[:12]
print(load_verified(good, expected))
try:
    load_verified(good.replace(b"0.7", b"0.8"), expected)
except RuntimeError as e:
    print("RuntimeError:", e)

['id,x', '1,0.5', '2,0.7']
RuntimeError: data changed: expected 2dc12095afc4, got ecce5d7da215


In [ ]:
def flatten(d, pre=""):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flatten(v, pre + k + "."))
        else:
            out[pre + k] = v
    return out
def cfg_diff(a, b):
    fa, fb = flatten(a), flatten(b)
    for k in sorted(set(fa) | set(fb)):
        if fa.get(k, "<missing>") != fb.get(k, "<missing>"):
            print(f"{k:18s} {fa.get(k, '<missing>')!s:10s} -> {fb.get(k, '<missing>')}")
old = {"model": {"depth": 4, "n_estimators": 20}, "train": {"seed": 1, "test_size": 0.3}}
new = {"model": {"depth": 4, "n_estimators": 100, "max_features": "sqrt"}, "train": {"seed": 1, "test_size": 0.2}}
cfg_diff(old, new)

model.max_features <missing>  -> sqrt
model.n_estimators 20         -> 100
train.test_size    0.3        -> 0.2


In [ ]:
import hashlib, json
cache = {}
calls = {"n": 0}
def cached_features(code_version, data_sha, cfg):
    k = hashlib.sha256(json.dumps([code_version, data_sha, cfg], sort_keys=True).encode()).hexdigest()[:10]
    if k in cache:
        return "hit  " + k
    calls["n"] += 1
    cache[k] = "features"
    return "miss " + k
print(cached_features("v1", "aa11", {"win": 7}))
print(cached_features("v1", "aa11", {"win": 7}))
print(cached_features("v1", "aa11", {"win": 14}))
print(cached_features("v2", "aa11", {"win": 7}))
print(cached_features("v1", "bb22", {"win": 7}))
print("computed", calls["n"], "times")

miss a2bed58901
hit  a2bed58901
miss e42630e999
miss c3f68ce4bc
miss 2164233276
computed 4 times


In [ ]:
ingredient = {"git": "code", "data_sha": "data", "config_sha": "config", "python": "environment",
              "numpy": "environment", "sklearn": "environment", "seed": "seed",
              "cpu": "hardware", "threads": "hardware"}
mine = {"git": "a1b2c3d", "data_sha": "dc64c3b65a7d", "config_sha": "2ccbe222bdf9", "python": "3.14.3",
        "numpy": "2.5.3", "sklearn": "1.9.1", "seed": 1, "cpu": "arm64", "threads": 8}
theirs = dict(mine, numpy="2.4.0", threads=2, data_sha="9f0e11aa22bb")
for k in mine:
    if mine[k] != theirs[k]:
        print(f"{ingredient[k]:12s} {k:9s} mine={mine[k]!s:13s} theirs={theirs[k]}")

data         data_sha  mine=dc64c3b65a7d  theirs=9f0e11aa22bb
environment  numpy     mine=2.5.3         theirs=2.4.0
hardware     threads   mine=8             theirs=2


## 2.13 Going Deeper: Bit-for-Bit Reproducibility and Its Limits

This part is for when you must go beyond "close enough": audits, regulated models, research claims, or a flaky test you cannot explain. Return to it after the basics feel easy.

In [ ]:
x1, x2 = 0.4, 0.4 + 1e-12
for step in range(1, 61):
    x1, x2 = 3.9 * x1 * (1 - x1), 3.9 * x2 * (1 - x2)
    if step in (10, 20, 30, 40, 50, 60):
        print(f"step {step:2d}  gap = {abs(x1 - x2):.2e}")

step 10  gap = 1.35e-10
step 20  gap = 5.21e-08
step 30  gap = 3.57e-05
step 40  gap = 1.24e-02
step 50  gap = 7.34e-02
step 60  gap = 2.16e-01


## 2.14 Mixed Drill and Self-Test

Do this without looking back. Each question mixes ingredients, like real incidents. Answers and a scoring guide follow the list.

## 2.15 Where This Connects

## Chapter recap

## Practice set

Problem 2.1 Runs give 0.90, 0.92, 0.91. Mean? **Answer:** (0.90 + 0.92 + 0.91) ÷ 3 = 2.73 ÷ 3 = 0.91.

In [ ]:
import hashlib
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
X, y = make_classification(n_samples=300, n_features=8, random_state=0)
def run(seed):
    p = RandomForestClassifier(n_estimators=10, random_state=seed).fit(X, y).predict_proba(X)
    return hashlib.sha256(p.tobytes()).hexdigest()
def distinct(seed, n=5):
    return len({run(seed) for _ in range(n)})
print("seed=None:", distinct(None), "distinct of 5")
print("seed=11  :", distinct(11), "distinct of 5")

seed=None: 5 distinct of 5
seed=11  : 1 distinct of 5


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/mlops/ch02.html